Installing dependencies and mounting Google Drive


In [ ]:
# C01 - Installing and mounting

# Install the packages that this notebook uses.
!pip install -q torch torchvision timm pandas scikit-learn matplotlib tqdm grad-cam scipy opencv-python medmnist

# Mount Google Drive.
from google.colab import drive
drive.mount("/content/drive")

print("Installed dependencies and mounted Google Drive.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 36.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 14.0 MB/s eta 0:00:00
Mounted at /content/drive
Installed dependencies and mounted Google Drive.


Importing all required libraries


In [ ]:
# C02 - Importing libraries

import os
import gc
import json
import time
import math
import random
import traceback
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    precision_recall_fscore_support,
    confusion_matrix,
    balanced_accuracy_score,
)

from PIL import Image
import cv2
from tqdm import tqdm
import timm

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

print("Imported libraries.")


Imported libraries.


Configuring smoke mode for quick tests


In [ ]:
# C03 - Configuring smoke mode

def configure_run_mode():
    """
    Configure smoke mode.
    Use smoke mode to test the full pipeline quickly.
    Turn it off before running the real benchmark.
    """
    global SMOKE_TEST
    global SMOKE_ONE_RUN_PER_DATASET
    global SMOKE_FORCE_FRACTION
    global SMOKE_MAX_EPOCHS
    global SMOKE_BATCH_SIZE
    global SMOKE_MAX_TRAIN_STEPS
    global SMOKE_MAX_EVAL_STEPS
    global SMOKE_GRADCAM_SAVE
    global SMOKE_MAX_SAMPLES_MESSIDOR2

    # Turn this off for the real experiments.
    SMOKE_TEST = False

    # Run only the first pending config row per dataset in smoke mode.
    SMOKE_ONE_RUN_PER_DATASET = False

    # Keep None to use the fraction from the config CSV.
    SMOKE_FORCE_FRACTION = None

    # Use small limits for quick pipeline checks.
    SMOKE_MAX_EPOCHS = 1
    SMOKE_BATCH_SIZE = 16
    SMOKE_MAX_TRAIN_STEPS = 6
    SMOKE_MAX_EVAL_STEPS = 6
    SMOKE_GRADCAM_SAVE = 6

    # Cap sample counts in smoke mode.
    SMOKE_MAX_SAMPLES_MESSIDOR2 = 800

    print("[RUN MODE] SMOKE_TEST:", SMOKE_TEST)
    print("[RUN MODE] SMOKE_ONE_RUN_PER_DATASET:", SMOKE_ONE_RUN_PER_DATASET)

configure_run_mode()


[RUN MODE] SMOKE_TEST: False
[RUN MODE] SMOKE_ONE_RUN_PER_DATASET: False


Configuring training settings and benchmark defaults


In [ ]:
# C04 - Configuring training settings

def configure_training():
    """
    Configure the core training settings.
    Keep these settings consistent across CNNs and transformers.
    """
    global IMG_SIZE
    global BATCH_SIZE
    global MAX_EPOCHS
    global EARLY_STOPPING_PATIENCE

    global BASE_LR
    global WEIGHT_DECAY
    global LR_PLATEAU_FACTOR
    global LR_PLATEAU_PATIENCE
    global LR_PLATEAU_MIN_LR

    global NUM_WORKERS
    global USE_AMP
    global FREE_TIER_MODE

    global DO_BORDER_CROP
    global DO_CLAHE

    global TRAIN_FRAC
    global VAL_FRAC
    global TEST_FRAC

    global LABEL_SMOOTHING
    global DROPOUT_P
    global BEST_DROPOUTS

    global USE_WEIGHTED_CROSS_ENTROPY
    global SAVE_BEST_WEIGHTS_ONLY
    global SAVE_PREDICTIONS_FOR_VAL
    global SAVE_PREDICTIONS_FOR_TEST
    global SAVE_SPLITS
    global SAVE_GRADCAM_RAW_ARRAYS
    global SAVE_GRADCAM
    global GRADCAM_NUM_SAVE_NORMAL

    global MIN_CLASS_COUNT_TO_KEEP_ISIC
    global MAX_PER_CLASS_TOTAL_ISIC

    IMG_SIZE = 224
    BATCH_SIZE = 32
    MAX_EPOCHS = 25
    EARLY_STOPPING_PATIENCE = 5

    # Use one learning rate rule for all main benchmark models.
    BASE_LR = 1e-4
    WEIGHT_DECAY = 1e-4
    LR_PLATEAU_FACTOR = 0.5
    LR_PLATEAU_PATIENCE = 3
    LR_PLATEAU_MIN_LR = 1e-6

    NUM_WORKERS = 2
    USE_AMP = True
    FREE_TIER_MODE = True

    # Apply the same image preprocessing policy inside each dataset pipeline.
    DO_BORDER_CROP = True
    DO_CLAHE = True

    TRAIN_FRAC = 0.60
    VAL_FRAC   = 0.20
    TEST_FRAC  = 0.20
    assert abs(TRAIN_FRAC + VAL_FRAC + TEST_FRAC - 1.0) < 1e-6

    LABEL_SMOOTHING = 0.05

    # Fallback dropout. Specific dataset/model values override this during each run.
    DROPOUT_P = 0.10

    # Best dropout values identified from the tuning runs.
    BEST_DROPOUTS = {
        ("ISIC2019", "efficientnet_b0"): 0.20,
        ("ISIC2019", "resnet18"): 0.50,
        ("MESSIDOR2", "efficientnet_b0"): 0.25,
        ("MESSIDOR2", "resnet18"): 0.25,
    }

    # Always use weighted cross-entropy in the benchmark.
    USE_WEIGHTED_CROSS_ENTROPY = True

    # Best-weight checkpoints are not needed for this version.
    SAVE_BEST_WEIGHTS_ONLY = False

    SAVE_PREDICTIONS_FOR_VAL = True
    SAVE_PREDICTIONS_FOR_TEST = True
    SAVE_SPLITS = True

    SAVE_GRADCAM = True
    SAVE_GRADCAM_RAW_ARRAYS = False
    GRADCAM_NUM_SAVE_NORMAL = 24

    # Use all ISIC classes and samples.
    MIN_CLASS_COUNT_TO_KEEP_ISIC = 1
    MAX_PER_CLASS_TOTAL_ISIC = None

    # Apply smoke overrides.
    if SMOKE_TEST:
        MAX_EPOCHS = SMOKE_MAX_EPOCHS
        BATCH_SIZE = SMOKE_BATCH_SIZE

    # Keep memory usage modest on Colab free tier.
    if FREE_TIER_MODE:
        NUM_WORKERS = 2
        SAVE_GRADCAM_RAW_ARRAYS = False

    print("[TRAINING] IMG_SIZE:", IMG_SIZE)
    print("[TRAINING] BATCH_SIZE:", BATCH_SIZE)
    print("[TRAINING] MAX_EPOCHS:", MAX_EPOCHS)
    print("[TRAINING] BASE_LR:", BASE_LR)
    print("[TRAINING] USE_WEIGHTED_CROSS_ENTROPY:", USE_WEIGHTED_CROSS_ENTROPY)
    print("[TRAINING] DEFAULT_DROPOUT_P:", DROPOUT_P)
    print("[TRAINING] BEST_DROPOUTS:", BEST_DROPOUTS)

configure_training()


[TRAINING] IMG_SIZE: 224
[TRAINING] BATCH_SIZE: 32
[TRAINING] MAX_EPOCHS: 25
[TRAINING] BASE_LR: 0.0001
[TRAINING] USE_WEIGHTED_CROSS_ENTROPY: True
[TRAINING] DEFAULT_DROPOUT_P: 0.1
[TRAINING] BEST_DROPOUTS: {('ISIC2019', 'efficientnet_b0'): 0.2, ('ISIC2019', 'resnet18'): 0.5, ('MESSIDOR2', 'efficientnet_b0'): 0.25, ('MESSIDOR2', 'resnet18'): 0.25}


In [ ]:
# C04B - Resolving per-dataset dropout values

def resolve_dropout_p(dataset_name, model_name, row_dropout=None, fallback_dropout=DROPOUT_P):
    """
    Resolve the dropout to use for one run.
    Priority:
    1. Explicit row value from the config CSV, if present
    2. Tuned dataset/model value from BEST_DROPOUTS
    3. Global fallback DROPOUT_P
    """
    if row_dropout is not None and not pd.isna(row_dropout):
        return float(row_dropout)

    key = (str(dataset_name).strip().upper(), str(model_name).strip())
    if key in BEST_DROPOUTS:
        return float(BEST_DROPOUTS[key])

    return float(fallback_dropout)

def apply_run_dropout(run_cfg, dataset_name):
    """
    Apply the resolved dropout to the global model-building setting before a run starts.
    """
    global DROPOUT_P

    run_dropout = resolve_dropout_p(
        dataset_name=dataset_name,
        model_name=run_cfg["model"],
        row_dropout=run_cfg.get("dropout_p", None),
        fallback_dropout=DROPOUT_P
    )

    DROPOUT_P = float(run_dropout)
    run_cfg["dropout_p"] = float(run_dropout)
    print(f"[DROPOUT] dataset={dataset_name} model={run_cfg['model']} dropout_p={DROPOUT_P:.2f}")
    return run_cfg

print("Prepared dropout helpers.")


Prepared dropout helpers.


Configuring Drive paths and output folders


In [ ]:
# C05 - Configuring paths

def configure_paths():
    """
    Configure paths for outputs and dataset caches.
    Keep results in Drive, but cache ISIC locally in the runtime.
    """
    global DRIVE_ROOT
    global CONFIG_PATH

    global RESULTS_ROOT
    global RUN_ARTIFACTS_DIR
    global MODELS_DIR
    global GRADCAM_DIR

    global ISIC_CACHE_DIR
    global MESSIDOR2_DIR

    DRIVE_ROOT = "/content/drive/MyDrive/MSC_Research/CNNs"

    CONFIG_PATH = f"{DRIVE_ROOT}/Configs/CNN_Config.csv"
    RESULTS_ROOT = f"{DRIVE_ROOT}/Results"
    RUN_ARTIFACTS_DIR = f"{RESULTS_ROOT}/Run_JSON"
    MODELS_DIR = f"{DRIVE_ROOT}/Models"
    GRADCAM_DIR = f"{DRIVE_ROOT}/Grad Cam"

    # Cache ISIC in the runtime's local storage only.
    ISIC_CACHE_DIR = "/content/local_datasets/ISIC2019"
    MESSIDOR2_DIR = f"{DRIVE_ROOT}/Messidor2"

    dirs_to_make = [
        DRIVE_ROOT,
        f"{DRIVE_ROOT}/Configs",
        RESULTS_ROOT,
        RUN_ARTIFACTS_DIR,
        MODELS_DIR,
        GRADCAM_DIR,
        ISIC_CACHE_DIR,
        MESSIDOR2_DIR,
    ]

    for d in dirs_to_make:
        os.makedirs(d, exist_ok=True)

    print("[PATHS] DRIVE_ROOT:", DRIVE_ROOT)
    print("[PATHS] CONFIG_PATH:", CONFIG_PATH)
    print("[PATHS] RUN_ARTIFACTS_DIR:", RUN_ARTIFACTS_DIR)
    print("[PATHS] ISIC_CACHE_DIR:", ISIC_CACHE_DIR)
    print("[PATHS] MESSIDOR2_DIR:", MESSIDOR2_DIR)

configure_paths()

[PATHS] DRIVE_ROOT: /content/drive/MyDrive/MSC_Research/CNNs
[PATHS] CONFIG_PATH: /content/drive/MyDrive/MSC_Research/CNNs/Configs/CNN_Config.csv
[PATHS] RUN_ARTIFACTS_DIR: /content/drive/MyDrive/MSC_Research/CNNs/Results/Run_JSON
[PATHS] ISIC_CACHE_DIR: /content/local_datasets/ISIC2019
[PATHS] MESSIDOR2_DIR: /content/drive/MyDrive/MSC_Research/CNNs/Messidor2


Loading the experiment config file


In [ ]:
# C06 - Loading the run config

def load_config():
    """
    Load the experiment config CSV.
    Add a status column if it does not exist.
    """
    global cfg

    cfg = pd.read_csv(CONFIG_PATH)
    cfg["dataset"] = cfg["dataset"].astype(str)

    if "status" not in cfg.columns:
        cfg["status"] = "Incomplete"

    cfg["status"] = cfg["status"].astype(str)

    print("[CONFIG] Loaded rows:", len(cfg))
    print("[CONFIG] Loaded columns:", len(cfg.columns))
    return cfg

cfg = load_config()


[CONFIG] Loaded rows: 36
[CONFIG] Loaded columns: 6


Tracking run progress across datasets


In [ ]:
# C07 - Tracking progress

def normalize_dataset_name(name: str):
    """Normalize a dataset name for safe comparisons."""
    return str(name).strip().upper()

def count_progress(cfg_df, dataset_name: str):
    """Count completed and pending runs for one dataset."""
    ds_target = normalize_dataset_name(dataset_name)
    ds_col = cfg_df["dataset"].astype(str).map(normalize_dataset_name)
    st_col = cfg_df["status"].astype(str).str.strip().str.lower()

    total = int((ds_col == ds_target).sum())
    complete = int(((ds_col == ds_target) & (st_col == "complete")).sum())
    pending = total - complete

    return {
        "dataset": dataset_name,
        "total": total,
        "complete": complete,
        "pending": pending,
    }

def print_progress(cfg_df, dataset_name: str):
    """Print progress for one dataset."""
    p = count_progress(cfg_df, dataset_name)
    print(f"[PROGRESS] {p['dataset']}: {p['complete']}/{p['total']} complete | {p['pending']} pending")

def print_overall_progress(cfg_df):
    """Print progress across all config rows."""
    st_col = cfg_df["status"].astype(str).str.strip().str.lower()
    total = len(cfg_df)
    complete = int((st_col == "complete").sum())
    pending = total - complete
    print(f"[PROGRESS] ALL: {complete}/{total} complete | {pending} pending")

print("Prepared progress helpers.")


Prepared progress helpers.


Building general utility functions


In [ ]:
# C08 - Building utility functions

def set_device_and_seed(seed: int):
    """
    Set the device and all random seeds.
    Use deterministic CuDNN settings for reproducibility.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    return device

def stratified_split_60_20_20(df, label_col="label", seed=42):
    """
    Split a dataframe into train, val, and test sets.
    Keep the class distribution similar across splits.
    """
    idx = np.arange(len(df))
    y = df[label_col].values

    sss1 = StratifiedShuffleSplit(n_splits=1, train_size=TRAIN_FRAC, random_state=seed)
    train_idx, tmp_idx = next(sss1.split(idx, y))

    df_train = df.iloc[train_idx].reset_index(drop=True)
    df_tmp   = df.iloc[tmp_idx].reset_index(drop=True)

    idx2 = np.arange(len(df_tmp))
    y2 = df_tmp[label_col].values

    sss2 = StratifiedShuffleSplit(n_splits=1, train_size=0.5, random_state=seed)
    val_idx, test_idx = next(sss2.split(idx2, y2))

    df_val  = df_tmp.iloc[val_idx].reset_index(drop=True)
    df_test = df_tmp.iloc[test_idx].reset_index(drop=True)

    return df_train, df_val, df_test

def take_fraction_stratified(df, frac, label_col="label", seed=42):
    """
    Take a stratified fraction of a dataframe.
    Keep the class distribution similar in the subset.
    """
    if frac >= 1.0 or len(df) == 0:
        return df.reset_index(drop=True)

    idx = np.arange(len(df))
    y = df[label_col].values

    sss = StratifiedShuffleSplit(n_splits=1, train_size=frac, random_state=seed)
    sub_idx, _ = next(sss.split(idx, y))

    return df.iloc[sub_idx].reset_index(drop=True)

def filter_classes_by_min_count(df, min_count, label_col="label", prefix=""):
    """
    Drop classes that do not have enough samples for stratified splitting.
    Print the kept and dropped classes for transparency.
    """
    df = df.copy()

    if len(df) == 0:
        print(f"{prefix} empty_dataframe")
        return df

    counts = df[label_col].value_counts().sort_index()
    keep_classes = counts[counts >= min_count].index.tolist()
    drop_classes = counts[counts < min_count].index.tolist()

    if len(drop_classes) > 0:
        print(f"{prefix} dropping_classes_below_min_count={min_count}: {drop_classes}")

    df = df[df[label_col].isin(keep_classes)].reset_index(drop=True)

    kept_counts = df[label_col].value_counts().sort_index().to_dict() if len(df) > 0 else {}
    print(f"{prefix} class_counts_after_filter={kept_counts}")

    return df

def hard_cleanup_after_run(*objects_to_del):
    """
    Clear Python and CUDA memory after a run.
    Use this after each experiment to reduce OOM issues.
    """
    try:
        for obj in objects_to_del:
            try:
                del obj
            except Exception:
                pass

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

    except Exception:
        pass

def count_model_parameters(model):
    """Count total and trainable parameters."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def json_safe(obj):
    """Convert NumPy values into JSON-safe Python types."""
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

def save_json(path, payload):
    """Save a Python object as JSON."""
    with open(path, "w") as f:
        json.dump(payload, f, indent=2, default=json_safe)

def append_log(log_path, text):
    """Append one line of text to a log file."""
    with open(log_path, "a") as f:
        f.write(str(text) + "\n")

def class_counts_dict(df, label_col="label"):
    """Return class counts as a dictionary."""
    return {
        int(k): int(v)
        for k, v in df[label_col].value_counts().sort_index().to_dict().items()
    }

def class_weights_from_labels(labels):
    """
    Compute class weights from the training labels only.
    Use inverse-frequency style weights for weighted cross-entropy.
    """
    labels = np.asarray(labels).astype(int)
    classes, counts = np.unique(labels, return_counts=True)

    total = counts.sum()
    num_classes = len(classes)

    weights = np.zeros(classes.max() + 1, dtype=np.float32)

    for c, cnt in zip(classes, counts):
        weights[c] = total / (num_classes * cnt)

    return torch.tensor(weights, dtype=torch.float32)

print("Prepared utility functions.")


Prepared utility functions.


Building artifact paths and save helpers


In [ ]:
# C09 - Building artifact paths and savers

def dataframe_to_records(rows_or_df):
    """
    Convert either a dataframe or row list into JSON-safe records.
    """
    if isinstance(rows_or_df, pd.DataFrame):
        df = rows_or_df.copy()
    else:
        df = pd.DataFrame(rows_or_df)

    if len(df) == 0:
        return []

    return json.loads(df.to_json(orient="records", date_format="iso"))

def save_dataframe_csv(path, rows_or_df):
    """Save either a dataframe or row list as CSV."""
    if isinstance(rows_or_df, pd.DataFrame):
        rows_or_df.to_csv(path, index=False)
    else:
        pd.DataFrame(rows_or_df).to_csv(path, index=False)
    return path

def save_model_weights(path, state_dict):
    """Save only the model weights."""
    torch.save(state_dict, path)
    return path

def build_split_records(df_split, dataset_name, split_name, run_id):
    """
    Convert one split dataframe into JSON-safe records.
    Add dataset, split, and run_id for easy reconstruction later.
    """
    df_out = df_split.copy()
    df_out["dataset"] = dataset_name
    df_out["split"] = split_name
    df_out["run_id"] = run_id
    return dataframe_to_records(df_out)

def get_run_paths(run_id):
    """
    Build file paths for one run.
    Keep the main summary CSV plus a single JSON bundle for results.
    """
    return {
        "final_csv": os.path.join(MODELS_DIR, f"{run_id}.csv"),
        "results_json": os.path.join(RUN_ARTIFACTS_DIR, f"{run_id}_results.json"),
        "gradcam_dir": os.path.join(GRADCAM_DIR, run_id),
    }

print("Prepared artifact helpers.")

Prepared artifact helpers.


Updating config status after successful runs


In [ ]:
# C10 - Updating config status

def mark_config_complete(cfg_df, run_id):
    """
    Mark one config row as complete.
    Skip writing changes in smoke mode.
    """
    if SMOKE_TEST:
        return cfg_df

    mask = cfg_df["run_id"] == run_id
    if mask.sum() == 0:
        return cfg_df

    cfg_df.loc[mask, "status"] = "Complete"
    cfg_df.to_csv(CONFIG_PATH, index=False)

    refreshed = pd.read_csv(CONFIG_PATH)
    refreshed["dataset"] = refreshed["dataset"].astype(str)
    refreshed["status"] = refreshed["status"].astype(str)

    return refreshed

print("Prepared config status helper.")


Prepared config status helper.


Evaluating models and collecting predictions


In [ ]:
# C11 - Evaluating models and collecting predictions

def evaluate(model, loader, device, desc="EVAL", return_predictions=False, split_name="unknown", run_id=""):
    """
    Evaluate a model on one loader.
    Save metrics and optionally collect prediction rows.
    """
    model.eval()

    # Use plain CE during evaluation.
    ce = nn.CrossEntropyLoss(label_smoothing=0.0)

    running_loss = 0.0
    n_seen = 0

    all_y = []
    all_pred = []
    all_prob = []

    prediction_rows = []

    with torch.no_grad():
        for step, batch in enumerate(tqdm(loader, desc=f"[{desc}]", leave=False)):
            if len(batch) == 3:
                x, y, meta = batch
            else:
                x, y = batch
                meta = None

            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            logits = model(x)
            loss = ce(logits, y)

            probs = torch.softmax(logits, dim=1)
            pred = probs.argmax(dim=1)

            running_loss += loss.item() * x.size(0)
            n_seen += x.size(0)

            y_np = y.cpu().numpy()
            pred_np = pred.cpu().numpy()
            prob_np = probs.cpu().numpy()

            all_y.extend(y_np.tolist())
            all_pred.extend(pred_np.tolist())
            all_prob.extend(prob_np.tolist())

            # Save one row per sample if requested.
            if return_predictions:
                for i in range(len(y_np)):
                    row = {
                        "run_id": run_id,
                        "split": split_name,
                        "sample_index_in_batch": i,
                        "true_label": int(y_np[i]),
                        "pred_label": int(pred_np[i]),
                        "correct": int(y_np[i] == pred_np[i]),
                        "max_prob": float(np.max(prob_np[i])),
                    }

                    # Save per-class probabilities.
                    for c in range(prob_np.shape[1]):
                        row[f"prob_class_{c}"] = float(prob_np[i][c])

                    # Save metadata fields if they exist.
                    if meta is not None and isinstance(meta, dict):
                        for k, v in meta.items():
                            try:
                                row[k] = v[i]
                            except Exception:
                                pass

                    prediction_rows.append(row)

            if SMOKE_TEST and (step + 1) >= SMOKE_MAX_EVAL_STEPS:
                break

    avg_loss = running_loss / max(1, n_seen)

    acc = accuracy_score(all_y, all_pred) if len(all_y) else float("nan")
    f1_macro = f1_score(all_y, all_pred, average="macro") if len(all_y) else float("nan")
    f1_weighted = f1_score(all_y, all_pred, average="weighted") if len(all_y) else float("nan")

    try:
        auroc_ovr = roc_auc_score(all_y, np.array(all_prob), multi_class="ovr")
    except Exception:
        auroc_ovr = float("nan")

    try:
        bal_acc = balanced_accuracy_score(all_y, all_pred)
    except Exception:
        bal_acc = float("nan")

    labels_sorted = sorted(np.unique(all_y).tolist()) if len(all_y) else []

    if len(all_y):
        prec, rec, f1_cls, support = precision_recall_fscore_support(
            all_y, all_pred, labels=labels_sorted, zero_division=0
        )
        cm = confusion_matrix(all_y, all_pred, labels=labels_sorted)
    else:
        prec, rec, f1_cls, support = [], [], [], []
        cm = np.array([[]])

    per_class_rows = []
    for i, cls in enumerate(labels_sorted):
        per_class_rows.append({
            "class_id": int(cls),
            "precision": float(prec[i]),
            "recall": float(rec[i]),
            "f1": float(f1_cls[i]),
            "support": int(support[i]),
        })

    return {
        "loss": avg_loss,
        "acc": acc,
        "balanced_acc": bal_acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
        "auroc_ovr": auroc_ovr,
        "n_seen": n_seen,
        "y_true": all_y,
        "y_pred": all_pred,
        "y_prob": all_prob,
        "prediction_rows": prediction_rows,
        "confusion_matrix": cm,
        "per_class_rows": per_class_rows,
    }

print("Prepared evaluation function.")


Prepared evaluation function.


Building Grad-CAM helpers


In [ ]:
# C12 - Building Grad-CAM helpers

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def denormalize_tensor_to_rgb_uint8(x_tensor):
    """
    Convert a normalized tensor back into an RGB uint8 image.
    Use this before saving Grad-CAM comparisons.
    """
    x = x_tensor.detach().cpu().numpy().transpose(1, 2, 0).astype(np.float32)
    x = (x * IMAGENET_STD) + IMAGENET_MEAN
    x = np.clip(x, 0, 1)
    return (x * 255.0).astype(np.uint8)

def get_target_layer(model, model_name):
    """
    Pick a suitable Grad-CAM target layer for the model.
    """
    mn = str(model_name).lower().strip()

    # Use the last residual block for ResNet.
    if mn.startswith("resnet") and hasattr(model, "layer4"):
        return model.layer4[-1]

    # Try common EfficientNet / timm endings.
    if hasattr(model, "conv_head"):
        return model.conv_head

    if hasattr(model, "blocks"):
        blocks = getattr(model, "blocks")
        if isinstance(blocks, (nn.ModuleList, list)) and len(blocks) > 0:
            return blocks[-1]

    if hasattr(model, "features"):
        feats = getattr(model, "features")
        if isinstance(feats, nn.Sequential) and len(feats) > 0:
            return feats[-1]

    # Fall back to the last Conv2d layer.
    last_conv = None
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            last_conv = m

    if last_conv is None:
        raise ValueError(f"Could not find a conv layer for Grad-CAM in '{model_name}'")

    return last_conv

def save_gradcam_comparisons(run_id, model, model_name, test_loader, device, num_to_save=24):
    """
    Save Grad-CAM before/after image pairs.
    Save a metadata CSV to link each pair to the sample.
    """
    out_dir = os.path.join(GRADCAM_DIR, run_id)
    os.makedirs(out_dir, exist_ok=True)

    metadata_rows = []
    metadata_path = os.path.join(out_dir, "gradcam_metadata.csv")

    model.eval()
    target_layer = get_target_layer(model, model_name)
    cam = GradCAM(model=model, target_layers=[target_layer])

    saved = 0

    for batch in tqdm(test_loader, desc="[GradCAM]", leave=False):
        if len(batch) == 3:
            batch_x, batch_y, batch_meta = batch
        else:
            batch_x, batch_y = batch
            batch_meta = None

        batch_x = batch_x.to(device, non_blocking=True)
        batch_y = batch_y.to(device, non_blocking=True)

        logits = model(batch_x)
        probs = torch.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

        for i in range(batch_x.size(0)):
            if saved >= num_to_save:
                pd.DataFrame(metadata_rows).to_csv(metadata_path, index=False)
                return out_dir

            x = batch_x[i:i+1]
            y = int(batch_y[i].item())
            pred = int(preds[i].item())
            conf = float(probs[i, pred].item())

            before_rgb = denormalize_tensor_to_rgb_uint8(batch_x[i])
            grayscale_cam = cam(input_tensor=x, targets=[ClassifierOutputTarget(y)])[0]

            before_float = before_rgb.astype(np.float32) / 255.0
            overlay = show_cam_on_image(before_float, grayscale_cam, use_rgb=True)

            before_path = os.path.join(out_dir, f"{saved:03d}_before.jpg")
            after_path  = os.path.join(out_dir, f"{saved:03d}_after.jpg")

            cv2.imwrite(before_path, cv2.cvtColor(before_rgb, cv2.COLOR_RGB2BGR))
            cv2.imwrite(after_path, cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

            if SAVE_GRADCAM_RAW_ARRAYS:
                npy_path = os.path.join(out_dir, f"{saved:03d}_cam.npy")
                np.save(npy_path, grayscale_cam)
            else:
                npy_path = ""

            row = {
                "run_id": run_id,
                "index": saved,
                "true_label": y,
                "pred_label": pred,
                "correct": int(y == pred),
                "confidence": conf,
                "before_path": before_path,
                "after_path": after_path,
                "cam_npy_path": npy_path,
            }

            if batch_meta is not None and isinstance(batch_meta, dict):
                for k, v in batch_meta.items():
                    try:
                        row[k] = v[i]
                    except Exception:
                        pass

            metadata_rows.append(row)
            saved += 1

    pd.DataFrame(metadata_rows).to_csv(metadata_path, index=False)
    return out_dir

print("Prepared Grad-CAM helpers.")


Prepared Grad-CAM helpers.


Building CNN models


In [ ]:
# C13 - Building CNN models

class ClassificationHead(nn.Module):
    """
    Build a simple classifier head.
    Apply dropout before the final linear layer.
    """
    def __init__(self, in_features, num_classes, dropout_p=0.0):
        super().__init__()
        self.head = nn.Sequential(
            nn.Dropout(p=float(dropout_p)),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.head(x)

def build_model(model_name: str, num_classes: int):
       mn = model_name.lower().strip()

    if mn in ["resnet18", "resnet34", "resnet50", "resnet101"]:
        fn = getattr(models, mn)
        model = fn(weights="IMAGENET1K_V1")
        in_features = model.fc.in_features
        model.fc = ClassificationHead(in_features, num_classes, dropout_p=DROPOUT_P)
        return model

    if "efficientnet" in mn:
        backbone = timm.create_model(mn, pretrained=True, num_classes=0)

        if not hasattr(backbone, "num_features"):
            raise ValueError(f"Model '{model_name}' does not expose num_features.")

        in_features = backbone.num_features

        if hasattr(backbone, "classifier"):
            backbone.classifier = ClassificationHead(in_features, num_classes, dropout_p=DROPOUT_P)
            return backbone

        if hasattr(backbone, "head"):
            backbone.head = ClassificationHead(in_features, num_classes, dropout_p=DROPOUT_P)
            return backbone

        raise ValueError(f"EfficientNet backbone '{model_name}' does not expose classifier/head.")

    raise ValueError(f"Unsupported model_name: {model_name}")

print("Prepared model builder.")


Prepared model builder.


Building image transforms


In [ ]:
# C14 - Building image transforms

train_tfms = transforms.Compose([
    # Resize images to the model input size.
    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    # Apply common augmentations for training.
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02),

    # Convert images to tensors and normalize them.
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN.tolist(), std=IMAGENET_STD.tolist()),
])

eval_tfms = transforms.Compose([
    # Keep evaluation deterministic.
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN.tolist(), std=IMAGENET_STD.tolist()),
])

print("Prepared transforms.")


Prepared transforms.


Building image preprocessing helpers


In [ ]:
# C15 - Building image preprocessing helpers

def border_crop(img_rgb: np.ndarray, thresh=10):
    """
    Crop black borders from a fundus or lesion image.
    Return the original image if no border region is found.
    """
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    _, mask = cv2.threshold(gray, thresh, 255, cv2.THRESH_BINARY)
    coords = cv2.findNonZero(mask)

    if coords is None:
        return img_rgb

    x, y, w, h = cv2.boundingRect(coords)
    return img_rgb[y:y+h, x:x+w]

def apply_clahe(img_rgb: np.ndarray):
    """
    Apply CLAHE to improve local contrast.
    Use LAB space to adjust the luminance channel only.
    """
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l2 = clahe.apply(l)

    lab2 = cv2.merge((l2, a, b))
    return cv2.cvtColor(lab2, cv2.COLOR_LAB2RGB)

print("Prepared preprocessing helpers.")


Prepared preprocessing helpers.


Building dataset wrappers and collate functions


In [ ]:
# C16 - Building dataset wrappers and collate functions

class MetadataImageDataset(Dataset):
    """
    Wrap a dataframe of image paths.
    Return image, label, and metadata for each sample.
    """
    def __init__(self, df, image_loader_fn, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_loader_fn = image_loader_fn
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self.image_loader_fn(row)

        if self.transform:
            img = self.transform(img)

        label = int(row["label"])

        meta = {
            "row_index": idx,
            "source_index": row["source_index"] if "source_index" in row else idx,
            "image_path": row["image_path"] if "image_path" in row else "",
            "dataset_sample_id": row["dataset_sample_id"] if "dataset_sample_id" in row else "",
        }

        if "image_id" in row:
            meta["image_id"] = row["image_id"]

        return img, label, meta

class ArrayMetadataDataset(Dataset):
    """
    Wrap an in-memory image array plus a dataframe.
    Use this for datasets backed by in-memory arrays.
    """
    def __init__(self, image_array, df, transform=None, index_col="idx"):
        self.image_array = image_array
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.index_col = index_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        arr_idx = int(row[self.index_col])

        img = Image.fromarray(self.image_array[arr_idx])

        if self.transform:
            img = self.transform(img)

        label = int(row["label"])

        meta = {
            "row_index": idx,
            "source_index": int(row["source_index"]) if "source_index" in row else arr_idx,
            "image_path": row["image_path"] if "image_path" in row else "",
            "dataset_sample_id": str(row["dataset_sample_id"]) if "dataset_sample_id" in row else str(arr_idx),
        }

        return img, label, meta

def collate_with_metadata(batch):
    """
    Collate a batch that includes metadata dictionaries.
    Stack images and labels, and keep metadata as lists.
    """
    xs, ys, metas = zip(*batch)

    x = torch.stack(xs, dim=0)
    y = torch.tensor(ys, dtype=torch.long)

    keys = set()
    for m in metas:
        keys.update(m.keys())

    meta_out = {}
    for k in keys:
        meta_out[k] = [m.get(k, "") for m in metas]

    return x, y, meta_out

print("Prepared dataset wrappers.")


Prepared dataset wrappers.


Building plain train loaders without oversampling


In [ ]:
# C17 - Building plain train loaders without oversampling

def build_plain_train_loader(dataset_obj, batch_size, num_workers, pin_memory=True):
    """
    Build a plain shuffled train loader.
    Do not use any rescue sampler or weighted sampler.
    """
    return DataLoader(
        dataset_obj,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
        collate_fn=collate_with_metadata,
    )

def build_eval_loader(dataset_obj, batch_size, num_workers, pin_memory=True):
    """
    Build a deterministic eval loader.
    Use this for validation and test sets.
    """
    return DataLoader(
        dataset_obj,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        collate_fn=collate_with_metadata,
    )

print("Prepared plain loader builders.")


Prepared plain loader builders.


Training one epoch


In [ ]:
# C18 - Training one epoch

def format_metric(v):
    """Format a metric for clean console printing."""
    try:
        if np.isnan(v):
            return "nan"
    except Exception:
        pass
    return f"{v:.4f}"

def assert_nonempty_loader(loader, split_name, run_id, dataset_name):
    """
    Raise an error if a loader is empty.
    Catch broken splits early.
    """
    try:
        next(iter(loader))
    except StopIteration:
        raise ValueError(f"[{dataset_name}][{run_id}] {split_name} loader is empty.")

def train_one_epoch(model, loader, device, optimizer, scaler, criterion, dataset_name, epoch):

    model.train()

    running_loss = 0.0
    n_seen = 0

    all_y = []
    all_pred = []

    t0 = time.time()

    for step, batch in enumerate(tqdm(loader, desc=f"[{dataset_name} TRAIN e{epoch}]", leave=False)):
        if len(batch) == 3:
            x, y, _ = batch
        else:
            x, y = batch

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=(USE_AMP and device == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        probs = torch.softmax(logits.detach(), dim=1)
        pred = probs.argmax(dim=1)

        running_loss += loss.item() * x.size(0)
        n_seen += x.size(0)

        all_y.extend(y.detach().cpu().numpy().tolist())
        all_pred.extend(pred.detach().cpu().numpy().tolist())

        if SMOKE_TEST and (step + 1) >= SMOKE_MAX_TRAIN_STEPS:
            break

    elapsed = time.time() - t0
    avg_loss = running_loss / max(1, n_seen)

    try:
        acc = accuracy_score(all_y, all_pred)
    except Exception:
        acc = float("nan")

    try:
        f1_macro = f1_score(all_y, all_pred, average="macro")
    except Exception:
        f1_macro = float("nan")

    return {
        "loss": avg_loss,
        "acc": acc,
        "f1_macro": f1_macro,
        "n_seen": n_seen,
        "seconds": elapsed,
    }

print("Prepared train_one_epoch.")


Prepared train_one_epoch.


Training one full run and saving all artifacts


In [ ]:
# C19 - Training one full run and saving all artifacts

def train_one_run(
    run_cfg,
    num_classes,
    train_loader,
    val_loader,
    test_loader,
    df_train_split,
    df_val_split,
    df_test_split,
    dataset_name,
    class_mapping_payload,
    run_index=None,
    run_total=None,
):
    """
    Train one run from start to finish.
    Save the main summary CSV plus one JSON bundle containing logs, history,
    predictions, splits, metrics, mappings, metadata, and a manifest.
    """
    global cfg

    run_id = run_cfg["run_id"]
    model_name = run_cfg["model"]
    seed = int(run_cfg["seed"])
    frac = float(run_cfg["data_fraction"])

    run_cfg = apply_run_dropout(run_cfg, dataset_name)
    dropout_p = float(run_cfg["dropout_p"])

    run_paths = get_run_paths(run_id)
    log_lines = []

    def log_line(text):
        text = str(text)
        log_lines.append(text)
        print(text)

    log_lines.append("=" * 80)
    log_lines.append(f"Run started: {datetime.now().isoformat()}")
    log_lines.append(
        f"run_id={run_id} dataset={dataset_name} model={model_name} "
        f"seed={seed} frac={frac} dropout_p={dropout_p}"
    )

    if run_index is not None and run_total is not None:
        prefix = f"[{dataset_name}] Run {run_index}/{run_total} run_id={run_id}"
    else:
        prefix = f"[{dataset_name}] run_id={run_id}"

    device = set_device_and_seed(seed)

    # Catch empty loaders before training.
    assert_nonempty_loader(train_loader, "train", run_id, dataset_name)
    assert_nonempty_loader(val_loader, "val", run_id, dataset_name)
    assert_nonempty_loader(test_loader, "test", run_id, dataset_name)

    # Build the model.
    model = build_model(model_name, num_classes=num_classes).to(device)
    total_params, trainable_params = count_model_parameters(model)

    optimizer = optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=LR_PLATEAU_FACTOR,
        patience=LR_PLATEAU_PATIENCE,
        min_lr=LR_PLATEAU_MIN_LR
    )

    class_weights_tensor = class_weights_from_labels(df_train_split["label"].values).to(device)
    class_weights_list = class_weights_tensor.detach().cpu().numpy().tolist()

    if USE_WEIGHTED_CROSS_ENTROPY:
        criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=LABEL_SMOOTHING)
    else:
        criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and device == "cuda"))

    best_val_loss = float("inf")
    best_epoch = -1
    best_state = None
    patience = 0
    stopped_early = False

    history_rows = []
    final_rows = []

    run_t0 = time.time()

    log_line(
        f"{prefix} start model={model_name} seed={seed} frac={frac:.2f} "
        f"dropout={dropout_p:.2f} lr={BASE_LR:g} device={device}"
    )

    for epoch in range(1, MAX_EPOCHS + 1):
        epoch_train = train_one_epoch(
            model=model,
            loader=train_loader,
            device=device,
            optimizer=optimizer,
            scaler=scaler,
            criterion=criterion,
            dataset_name=dataset_name,
            epoch=epoch
        )

        val_metrics = evaluate(
            model,
            val_loader,
            device,
            desc=f"{dataset_name} VAL",
            return_predictions=SAVE_PREDICTIONS_FOR_VAL,
            split_name="val",
            run_id=run_id
        )

        val_loss = val_metrics["loss"]
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]["lr"]

        improved = val_loss < (best_val_loss - 1e-6)

        if improved:
            best_val_loss = val_loss
            best_epoch = epoch
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1

        history_row = {
            "epoch": epoch,
            "run_id": run_id,
            "dataset": dataset_name,
            "model": model_name,
            "seed": seed,
            "data_fraction_requested": float(run_cfg["data_fraction"]),
            "data_fraction_used": frac,

            "train_loss": epoch_train["loss"],
            "train_acc": epoch_train["acc"],
            "train_f1_macro": epoch_train["f1_macro"],
            "train_n_seen": epoch_train["n_seen"],
            "train_seconds": epoch_train["seconds"],

            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["acc"],
            "val_balanced_acc": val_metrics["balanced_acc"],
            "val_f1_macro": val_metrics["f1_macro"],
            "val_f1_weighted": val_metrics["f1_weighted"],
            "val_auroc_ovr": val_metrics["auroc_ovr"],
            "val_n_seen": val_metrics["n_seen"],

            "lr": current_lr,
            "best_val_loss_so_far": best_val_loss,
            "best_epoch_so_far": best_epoch,
            "patience_counter": patience,
        }
        history_rows.append(history_row)

        msg = (
            f"{prefix} e{epoch}/{MAX_EPOCHS} "
            f"train={format_metric(epoch_train['loss'])} "
            f"train_acc={format_metric(epoch_train['acc'])} "
            f"val_loss={format_metric(val_metrics['loss'])} "
            f"val_acc={format_metric(val_metrics['acc'])} "
            f"val_f1={format_metric(val_metrics['f1_macro'])} "
            f"val_auc={format_metric(val_metrics['auroc_ovr'])} "
            f"lr={current_lr:g} "
            f"pat={patience}/{EARLY_STOPPING_PATIENCE}"
        )
        log_line(msg)

        if patience >= EARLY_STOPPING_PATIENCE:
            stopped_early = True
            log_line(f"{prefix} early_stop")
            break

        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    val_final_metrics = evaluate(
        model,
        val_loader,
        device,
        desc=f"{dataset_name} VAL_FINAL",
        return_predictions=SAVE_PREDICTIONS_FOR_VAL,
        split_name="val",
        run_id=run_id
    )

    test_metrics = evaluate(
        model,
        test_loader,
        device,
        desc=f"{dataset_name} TEST",
        return_predictions=SAVE_PREDICTIONS_FOR_TEST,
        split_name="test",
        run_id=run_id
    )

    run_seconds = time.time() - run_t0

    final_rows.append({
        "run_id": run_id,
        "dataset": dataset_name,
        "model": model_name,
        "seed": seed,

        "data_fraction_requested": float(run_cfg["data_fraction"]),
        "data_fraction_used": frac,

        "num_classes": num_classes,
        "best_epoch": best_epoch,
        "epochs_trained": len(history_rows),
        "stopped_early": int(stopped_early),
        "best_val_loss": best_val_loss,

        "final_val_loss": val_final_metrics["loss"],
        "final_val_acc": val_final_metrics["acc"],
        "final_val_balanced_acc": val_final_metrics["balanced_acc"],
        "final_val_f1_macro": val_final_metrics["f1_macro"],
        "final_val_f1_weighted": val_final_metrics["f1_weighted"],
        "final_val_auroc_ovr": val_final_metrics["auroc_ovr"],

        "test_loss": test_metrics["loss"],
        "test_acc": test_metrics["acc"],
        "test_balanced_acc": test_metrics["balanced_acc"],
        "test_f1_macro": test_metrics["f1_macro"],
        "test_f1_weighted": test_metrics["f1_weighted"],
        "test_auroc_ovr": test_metrics["auroc_ovr"],

        "total_params": total_params,
        "trainable_params": trainable_params,
        "run_seconds": run_seconds,

        "device": device,
        "batch_size": BATCH_SIZE,
        "img_size": IMG_SIZE,
        "dropout_p": dropout_p,
        "label_smoothing": LABEL_SMOOTHING,
        "weight_decay": WEIGHT_DECAY,
        "base_lr": BASE_LR,
        "use_amp": int(USE_AMP),
        "use_weighted_cross_entropy": int(USE_WEIGHTED_CROSS_ENTROPY),
        "do_border_crop": int(DO_BORDER_CROP),
        "do_clahe": int(DO_CLAHE),
        "smoke_test": int(SMOKE_TEST),
    })

    # Keep a lightweight summary CSV in Models for quick comparisons.
    save_dataframe_csv(run_paths["final_csv"], final_rows)

    metadata = {
        "run_id": run_id,
        "dataset": dataset_name,
        "model": model_name,
        "seed": seed,

        "data_fraction_requested": float(run_cfg["data_fraction"]),
        "data_fraction_used": frac,

        "num_classes": num_classes,
        "device": device,
        "img_size": IMG_SIZE,
        "batch_size": BATCH_SIZE,
        "max_epochs": MAX_EPOCHS,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,

        "base_lr": BASE_LR,
        "weight_decay": WEIGHT_DECAY,
        "label_smoothing": LABEL_SMOOTHING,
        "dropout_p": dropout_p,

        "use_amp": USE_AMP,
        "use_weighted_cross_entropy": USE_WEIGHTED_CROSS_ENTROPY,
        "class_weights_used": class_weights_list,

        "do_border_crop": DO_BORDER_CROP,
        "do_clahe": DO_CLAHE,
        "imagenet_mean": IMAGENET_MEAN.tolist(),
        "imagenet_std": IMAGENET_STD.tolist(),

        "total_params": total_params,
        "trainable_params": trainable_params,

        "train_class_counts": class_counts_dict(df_train_split),
        "val_class_counts": class_counts_dict(df_val_split),
        "test_class_counts": class_counts_dict(df_test_split),

        "best_epoch": best_epoch,
        "epochs_trained": len(history_rows),
        "stopped_early": stopped_early,
        "best_val_loss": best_val_loss,
        "run_seconds": run_seconds,

        "smoke_test": SMOKE_TEST,
    }

    if SAVE_GRADCAM:
        num_cam = SMOKE_GRADCAM_SAVE if SMOKE_TEST else GRADCAM_NUM_SAVE_NORMAL

        gradcam_path = save_gradcam_comparisons(
            run_id=run_id,
            model=model,
            model_name=model_name,
            test_loader=test_loader,
            device=device,
            num_to_save=num_cam
        )
    else:
        gradcam_path = ""

    manifest = {
        "run_id": run_id,
        "dataset": dataset_name,
        "model": model_name,
        "final_csv": run_paths["final_csv"],
        "results_json": run_paths["results_json"],
        "gradcam_dir": gradcam_path,
    }

    results_bundle = {
        "run_id": run_id,
        "dataset": dataset_name,
        "model": model_name,
        "final_summary": dataframe_to_records(final_rows),
        "history": history_rows,
        "logs": log_lines,
        "metadata": metadata,
        "class_mapping": class_mapping_payload,
        "predictions": {
            "val": val_final_metrics["prediction_rows"] if SAVE_PREDICTIONS_FOR_VAL else [],
            "test": test_metrics["prediction_rows"] if SAVE_PREDICTIONS_FOR_TEST else [],
        },
        "splits": {
            "train": build_split_records(df_train_split, dataset_name, "train", run_id) if SAVE_SPLITS else [],
            "val": build_split_records(df_val_split, dataset_name, "val", run_id) if SAVE_SPLITS else [],
            "test": build_split_records(df_test_split, dataset_name, "test", run_id) if SAVE_SPLITS else [],
        },
        "metrics": {
            "per_class_test": test_metrics["per_class_rows"],
            "confusion_matrix_test": test_metrics["confusion_matrix"],
            "val_final": val_final_metrics,
            "test_final": test_metrics,
        },
        "manifest": manifest,
    }

    save_json(run_paths["results_json"], results_bundle)

    log_line(
        f"{prefix} done "
        f"test_acc={format_metric(test_metrics['acc'])} "
        f"test_f1={format_metric(test_metrics['f1_macro'])} "
        f"test_auc={format_metric(test_metrics['auroc_ovr'])}"
    )

    cfg = mark_config_complete(cfg, run_id)

    hard_cleanup_after_run(
        model,
        optimizer,
        scheduler,
        scaler,
        class_weights_tensor,
        best_state,
        train_loader,
        val_loader,
        test_loader
    )

Building shared dataset run helpers


In [ ]:
# C20 - Building shared dataset run helpers

def run_fractioned_split_experiment(
    cfg_df,
    dataset_name,
    df_runs,
    df_pool_full,
    dataset_builder_fn,
    class_mapping_payload_builder_fn,
    num_classes_getter_fn=None,
    run_head_in_smoke=True,
):
    """
    Run a set of config rows for a dataset that uses one pooled dataframe.
    Use this for pooled dataframe datasets.
    """
    if len(df_runs) == 0:
        print(f"[{dataset_name}] pending=0")
        return

    if SMOKE_TEST and SMOKE_ONE_RUN_PER_DATASET and run_head_in_smoke:
        df_runs = df_runs.head(1).copy()

    if len(df_pool_full) == 0 or df_pool_full["label"].nunique() < 2:
        print(f"[{dataset_name}] no_usable_pool")
        return

    print(f"[{dataset_name}] runs:", len(df_runs), "| pool_size:", len(df_pool_full))

    for i, (_, row) in enumerate(df_runs.iterrows(), start=1):
        run_cfg = {
            "run_id": row["run_id"],
            "model": row["model"],
            "data_fraction": float(row["data_fraction"]),
            "seed": int(row["seed"]),
            "dropout_p": row["dropout"] if "dropout" in row.index else np.nan,
        }

        frac = float(run_cfg["data_fraction"])
        if SMOKE_TEST and (SMOKE_FORCE_FRACTION is not None):
            frac = float(SMOKE_FORCE_FRACTION)
        run_cfg["data_fraction"] = frac

        # Take the requested stratified fraction.
        df_pool = take_fraction_stratified(
            df_pool_full,
            frac=frac,
            label_col="label",
            seed=run_cfg["seed"]
        )

        print(f"[{dataset_name}] run_id={run_cfg['run_id']} size_after_fraction={len(df_pool)}")

        # Drop classes that are too small for 60/20/20 stratified splitting.
        # Use 5 as a safe minimum across runs.
        df_pool = filter_classes_by_min_count(
            df_pool,
            min_count=5,
            label_col="label",
            prefix=f"[{dataset_name}][{run_cfg['run_id']}]"
        )

        if len(df_pool) == 0 or df_pool["label"].nunique() < 2:
            print(f"[{dataset_name}] skip run_id: {run_cfg['run_id']} reason=too_small_after_class_filter")
            continue

        # Split into train, val, and test.
        try:
            df_train, df_val, df_test = stratified_split_60_20_20(
                df_pool,
                label_col="label",
                seed=run_cfg["seed"]
            )
        except Exception as e:
            print(f"[{dataset_name}] skip run_id: {run_cfg['run_id']} reason=split_failed {repr(e)}")
            continue

        # Build datasets and dataloaders.
        ds_tr, ds_va, ds_te = dataset_builder_fn(df_train, df_val, df_test)

        train_loader = build_plain_train_loader(ds_tr, BATCH_SIZE, NUM_WORKERS, pin_memory=True)
        val_loader = build_eval_loader(ds_va, BATCH_SIZE, NUM_WORKERS, pin_memory=True)
        test_loader = build_eval_loader(ds_te, BATCH_SIZE, NUM_WORKERS, pin_memory=True)

        if num_classes_getter_fn is None:
            num_classes = int(df_pool["label"].nunique())
        else:
            num_classes = int(num_classes_getter_fn(df_pool, df_train, df_val, df_test))

        class_mapping_payload = class_mapping_payload_builder_fn(df_pool, df_train, df_val, df_test)

        try:
            train_one_run(
                run_cfg=run_cfg,
                num_classes=num_classes,
                train_loader=train_loader,
                val_loader=val_loader,
                test_loader=test_loader,
                df_train_split=df_train,
                df_val_split=df_val,
                df_test_split=df_test,
                dataset_name=dataset_name,
                class_mapping_payload=class_mapping_payload,
                run_index=i,
                run_total=len(df_runs)
            )
        except Exception as e:
            print(f"[{dataset_name}] run_error run_id: {run_cfg['run_id']} | {repr(e)}")
            traceback.print_exc()
            hard_cleanup_after_run(train_loader, val_loader, test_loader)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        print_progress(cfg, dataset_name)
        print_overall_progress(cfg)

print("Prepared shared dataset run helpers.")


Prepared shared dataset run helpers.


Building the ISIC2019 pipeline


In [ ]:
# C21 - Building the ISIC2019 pipeline

def ensure_isic2019_cached():
    """
    Download and cache ISIC2019 files in the local Colab runtime if they are missing.
    """
    train_dir = f"{ISIC_CACHE_DIR}/ISIC_2019_Training_Input"
    test_dir  = f"{ISIC_CACHE_DIR}/ISIC_2019_Test_Input"
    train_csv = f"{ISIC_CACHE_DIR}/ISIC_2019_Training_GroundTruth.csv"
    test_csv  = f"{ISIC_CACHE_DIR}/ISIC_2019_Test_GroundTruth.csv"

    if all(os.path.exists(p) for p in [train_dir, test_dir, train_csv, test_csv]):
        print("[ISIC] cached=true")
        return train_dir, test_dir, train_csv, test_csv

    print("[ISIC] caching_to_local_runtime")
    os.makedirs(ISIC_CACHE_DIR, exist_ok=True)

    train_zip = f"{ISIC_CACHE_DIR}/ISIC_2019_Training_Input.zip"
    test_zip  = f"{ISIC_CACHE_DIR}/ISIC_2019_Test_Input.zip"

    if not os.path.exists(train_zip):
        !curl -L -o "{train_zip}" "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_Input.zip"

    if not os.path.exists(test_zip):
        !curl -L -o "{test_zip}" "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Test_Input.zip"

    if not os.path.exists(train_csv):
        !wget -q -O "{train_csv}" "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_GroundTruth.csv"

    if not os.path.exists(test_csv):
        !wget -q -O "{test_csv}" "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Test_GroundTruth.csv"

    if not os.path.exists(train_dir):
        !unzip -q "{train_zip}" -d "{ISIC_CACHE_DIR}"

    if not os.path.exists(test_dir):
        !unzip -q "{test_zip}" -d "{ISIC_CACHE_DIR}"

    print("[ISIC] cached=now")
    return train_dir, test_dir, train_csv, test_csv

def load_isic_train_df(train_csv_path, train_img_dir, min_count=MIN_CLASS_COUNT_TO_KEEP_ISIC, cap_per_class=MAX_PER_CLASS_TOTAL_ISIC):
    """
    Load the ISIC training dataframe.
    Keep all available classes and samples.
    """
    df = pd.read_csv(train_csv_path)

    label_cols = df.columns[1:]
    df["label_name"] = df[label_cols].idxmax(axis=1)

    df["image_id"] = df["image"].astype(str)
    df["image_path"] = df["image_id"].apply(lambda x: os.path.join(train_img_dir, f"{x}.jpg"))

    # Keep only rows whose images exist.
    df = df[df["image_path"].apply(os.path.exists)].reset_index(drop=True)

    # Keep all available classes and samples.
    if min_count is not None and min_count > 1:
        counts = df["label_name"].value_counts()
        keep_classes = counts[counts >= min_count].index
        df = df[df["label_name"].isin(keep_classes)].reset_index(drop=True)

    if cap_per_class is not None:
        capped_parts = []
        for cls_name, g in df.groupby("label_name"):
            n_take = min(len(g), cap_per_class)
            capped_parts.append(g.sample(n=n_take, random_state=0))

        df = pd.concat(capped_parts, ignore_index=True)

    # Build a new contiguous label mapping.
    classes = sorted(df["label_name"].unique().tolist())
    label_map = {name: i for i, name in enumerate(classes)}

    df["label"] = df["label_name"].map(label_map).astype(int)
    df["source_index"] = np.arange(len(df))
    df["dataset_sample_id"] = df["image_id"]

    return df, label_map

def load_isic_test_df_mapped(test_csv_path, test_img_dir, label_map):
    """
    Load the ISIC test dataframe and map it to the training label map.
    Keep only classes present in the training map.
    """
    df = pd.read_csv(test_csv_path)

    label_cols = df.columns[1:]
    df["label_name"] = df[label_cols].idxmax(axis=1)

    df["image_id"] = df["image"].astype(str)
    df["image_path"] = df["image_id"].apply(lambda x: os.path.join(test_img_dir, f"{x}.jpg"))

    df = df[df["image_path"].apply(os.path.exists)].reset_index(drop=True)
    df = df[df["label_name"].isin(label_map)].copy()

    df["label"] = df["label_name"].map(label_map).astype(int)
    df["source_index"] = np.arange(len(df))
    df["dataset_sample_id"] = df["image_id"]

    return df.reset_index(drop=True)

def isic_loader_fn(row):
    """
    Load one ISIC image and apply optional preprocessing.
    """
    path = row["image_path"]
    img = Image.open(path).convert("RGB")
    img_np = np.array(img)

    if DO_BORDER_CROP:
        img_np = border_crop(img_np)

    if DO_CLAHE:
        img_np = apply_clahe(img_np)

    return Image.fromarray(img_np)

def run_isic2019_from_config(cfg_df):
    """
    Run all pending ISIC2019 config rows.
    Use the official training set for train/val and the official test set for testing.
    """
    print_progress(cfg_df, "ISIC2019")

    df_runs = cfg_df[
        (cfg_df["dataset"].astype(str).str.upper().str.strip() == "ISIC2019") &
        (cfg_df["status"].astype(str).str.lower().str.strip() != "complete")
    ].copy()

    if len(df_runs) == 0:
        print("[ISIC] pending=0")
        return

    if SMOKE_TEST and SMOKE_ONE_RUN_PER_DATASET:
        df_runs = df_runs.head(1).copy()

    train_dir, test_dir, train_csv, test_csv = ensure_isic2019_cached()

    df_train_full, label_map = load_isic_train_df(train_csv, train_dir)
    df_test_full = load_isic_test_df_mapped(test_csv, test_dir, label_map)

    num_classes = len(label_map)

    print("[ISIC] num_classes:", num_classes)
    print("[ISIC] filtered_train_size:", len(df_train_full))
    print("[ISIC] official_test_size:", len(df_test_full))
    print("[ISIC] runs:", len(df_runs))

    for i, (_, row) in enumerate(df_runs.iterrows(), start=1):
        run_cfg = {
            "run_id": row["run_id"],
            "model": row["model"],
            "data_fraction": float(row["data_fraction"]),
            "seed": int(row["seed"]),
            "dropout_p": row["dropout"] if "dropout" in row.index else np.nan,
        }

        frac = float(run_cfg["data_fraction"])
        if SMOKE_TEST and (SMOKE_FORCE_FRACTION is not None):
            frac = float(SMOKE_FORCE_FRACTION)
        run_cfg["data_fraction"] = frac

        # Take a fraction of the official training set only.
        df_frac = take_fraction_stratified(
            df_train_full,
            frac=frac,
            label_col="label",
            seed=run_cfg["seed"]
        )

        # Split the selected training subset into train and val.
        sss = StratifiedShuffleSplit(n_splits=1, train_size=0.8, random_state=run_cfg["seed"])
        idx = np.arange(len(df_frac))
        y = df_frac["label"].values
        tr_idx, va_idx = next(sss.split(idx, y))

        df_tr = df_frac.iloc[tr_idx].reset_index(drop=True)
        df_va = df_frac.iloc[va_idx].reset_index(drop=True)
        df_te = df_test_full.copy().reset_index(drop=True)

        # Build datasets.
        ds_tr = MetadataImageDataset(df_tr, image_loader_fn=isic_loader_fn, transform=train_tfms)
        ds_va = MetadataImageDataset(df_va, image_loader_fn=isic_loader_fn, transform=eval_tfms)
        ds_te = MetadataImageDataset(df_te, image_loader_fn=isic_loader_fn, transform=eval_tfms)

        # Build loaders.
        train_loader = build_plain_train_loader(ds_tr, BATCH_SIZE, NUM_WORKERS, pin_memory=True)
        val_loader = build_eval_loader(ds_va, BATCH_SIZE, NUM_WORKERS, pin_memory=True)
        test_loader = build_eval_loader(ds_te, BATCH_SIZE, NUM_WORKERS, pin_memory=True)

        class_mapping_payload = {
            "dataset": "ISIC2019",
            "label_map_name_to_id": label_map,
            "label_map_id_to_name": {str(v): k for k, v in label_map.items()},
            "notes": {
                "min_class_count_to_keep": MIN_CLASS_COUNT_TO_KEEP_ISIC,
                "max_per_class_total": MAX_PER_CLASS_TOTAL_ISIC,
            }
        }

        try:
            train_one_run(
                run_cfg=run_cfg,
                num_classes=num_classes,
                train_loader=train_loader,
                val_loader=val_loader,
                test_loader=test_loader,
                df_train_split=df_tr,
                df_val_split=df_va,
                df_test_split=df_te,
                dataset_name="ISIC2019",
                class_mapping_payload=class_mapping_payload,
                run_index=i,
                run_total=len(df_runs)
            )
        except Exception as e:
            print("[ISIC] run_error run_id:", run_cfg["run_id"], "|", repr(e))
            traceback.print_exc()
            hard_cleanup_after_run(train_loader, val_loader, test_loader)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        print_progress(cfg, "ISIC2019")
        print_overall_progress(cfg)

print("Prepared ISIC2019 pipeline.")


Prepared ISIC2019 pipeline.


Building the Messidor2 pipeline


In [ ]:
# C23 - Building the Messidor2 pipeline

def load_messidor2_adjudicated_df():
    """
    Load the local Messidor2 dataframe from Drive.
    Keep only gradable images with adjudicated DR grades.
    """
    images_dir = f"{MESSIDOR2_DIR}/IMAGES"
    csv_path = f"{MESSIDOR2_DIR}/messidor_data.csv"

    df = pd.read_csv(csv_path)

    # Keep only gradable rows with a usable adjudicated label.
    df = df[df["adjudicated_gradable"] == 1].copy()
    df = df.dropna(subset=["adjudicated_dr_grade"]).copy()

    df["label"] = df["adjudicated_dr_grade"].astype(int)
    df["image_id"] = df["image_id"].astype(str)

    # Keep the same image path rule as before.
    df["image_path"] = df["image_id"].apply(lambda x: os.path.join(images_dir, str(x)))

    # Keep only rows whose image file exists.
    df = df[df["image_path"].apply(os.path.exists)].reset_index(drop=True)

    df["source_index"] = np.arange(len(df))
    df["dataset_sample_id"] = df["image_id"]

    # Build a stratified smoke subset instead of a random one.
    if SMOKE_TEST:
        smoke_n = min(len(df), SMOKE_MAX_SAMPLES_MESSIDOR2)
        smoke_frac = smoke_n / len(df) if len(df) > 0 else 1.0

        if smoke_frac < 1.0:
            df = take_fraction_stratified(
                df,
                frac=smoke_frac,
                label_col="label",
                seed=0
            )

        print("[Messidor2] smoke_subset_size:", len(df))
        print("[Messidor2] smoke_class_counts:", class_counts_dict(df))

    return df.reset_index(drop=True)

def messidor2_loader_fn(row):
    """
    Load one Messidor2 image and apply optional preprocessing.
    """
    path = row["image_path"]
    img = Image.open(path).convert("RGB")
    img_np = np.array(img)

    if DO_BORDER_CROP:
        img_np = border_crop(img_np)

    if DO_CLAHE:
        img_np = apply_clahe(img_np)

    return Image.fromarray(img_np)

def build_messidor2_datasets(df_train, df_val, df_test):
    """
    Build Messidor2 train, val, and test datasets.
    """
    ds_tr = MetadataImageDataset(df_train, image_loader_fn=messidor2_loader_fn, transform=train_tfms)
    ds_va = MetadataImageDataset(df_val, image_loader_fn=messidor2_loader_fn, transform=eval_tfms)
    ds_te = MetadataImageDataset(df_test, image_loader_fn=messidor2_loader_fn, transform=eval_tfms)
    return ds_tr, ds_va, ds_te

def build_messidor2_class_mapping_payload(df_pool, df_train, df_val, df_test):
    """
    Build class mapping info for Messidor2.
    """
    unique_labels = sorted(df_pool["label"].unique().tolist())

    return {
        "dataset": "Messidor2",
        "label_map_name_to_id": {},
        "label_map_id_to_name": {str(x): str(x) for x in unique_labels},
        "notes": {
            "source": "Messidor2 adjudicated DR grades are used as integer labels."
        }
    }

def run_messidor2_from_config(cfg_df):
    """
    Run all pending Messidor2 config rows.
    """
    print_progress(cfg_df, "Messidor2")

    ds_upper = cfg_df["dataset"].astype(str).str.upper().str.strip()

    df_runs = cfg_df[
        (ds_upper == "MESSIDOR2") &
        (cfg_df["status"].astype(str).str.lower().str.strip() != "complete")
    ].copy()

    if len(df_runs) == 0:
        print("[Messidor2] pending=0")
        return

    df_full_all = load_messidor2_adjudicated_df()

    run_fractioned_split_experiment(
        cfg_df=cfg_df,
        dataset_name="Messidor2",
        df_runs=df_runs,
        df_pool_full=df_full_all,
        dataset_builder_fn=build_messidor2_datasets,
        class_mapping_payload_builder_fn=build_messidor2_class_mapping_payload,
        num_classes_getter_fn=lambda df_pool, *_: df_pool["label"].nunique(),
        run_head_in_smoke=True,
    )

print("Prepared Messidor2 pipeline.")


Prepared Messidor2 pipeline.


Building the master runner


In [ ]:
# C24 - Building the master runner

def run_all_selected_datasets(dataset_name, cfg_df=None):
    """
    Run all pending config rows for one selected dataset.
    """
    cfg_df = cfg if cfg_df is None else cfg_df
    ds = str(dataset_name).strip().upper()

    print_overall_progress(cfg_df)

    if ds == "ISIC2019":
        run_isic2019_from_config(cfg_df)

    elif ds == "MESSIDOR2":
        run_messidor2_from_config(cfg_df)

    else:
        raise ValueError("Unknown dataset_name. Use: ISIC2019 or Messidor2")

    print_overall_progress(cfg_df)

def build_dataset_calls(dataset_name):
    """
    Build a dataset-specific config slice and attach the tuned dropout values.
    This keeps the launch calls clean while still applying the proper dropout internally.
    """
    ds = str(dataset_name).strip().upper()

    df = cfg.copy()
    df["dataset_upper"] = df["dataset"].astype(str).str.upper().str.strip()
    df["model"] = df["model"].astype(str).str.strip()

    df = df[df["dataset_upper"] == ds].copy()
    if len(df) == 0:
        raise ValueError(f"No config rows found for dataset: {dataset_name}")

    df["dropout"] = df.apply(
        lambda row: resolve_dropout_p(
            dataset_name=row["dataset"],
            model_name=row["model"],
            row_dropout=row["dropout"] if "dropout" in row.index else np.nan,
            fallback_dropout=DROPOUT_P
        ),
        axis=1
    )

    cols = [c for c in df.columns if c != "dataset_upper"]
    df = df[cols].copy()

    print(f"[DATASET_CALLS] dataset={ds} rows={len(df)}")
    print(df[["dataset", "model", "run_id", "dropout"]].drop_duplicates().to_string(index=False))
    return df

def run_isic_calls():
    """
    Run only the ISIC2019 rows.
    ISIC is cached locally in /content, so one download per runtime is enough.
    """
    run_all_selected_datasets("ISIC2019", cfg_df=build_dataset_calls("ISIC2019"))

def run_messidor_calls():
    """
    Run only the Messidor2 rows.
    """
    run_all_selected_datasets("MESSIDOR2", cfg_df=build_dataset_calls("MESSIDOR2"))

print("Prepared master runner.")
print("Call: run_isic_calls()")
print("Call: run_messidor_calls()")


Prepared master runner.
Call: run_isic_calls()
Call: run_messidor_calls()


In [ ]:
# C25 - Running ISIC2019 experiments

run_isic_calls()


[DATASET_CALLS] dataset=ISIC2019 rows=18
 dataset           model                          run_id  dropout
ISIC2019        resnet18         ISIC2019_resnet18_25_42      0.5
ISIC2019        resnet18         ISIC2019_resnet18_25_43      0.5
ISIC2019        resnet18         ISIC2019_resnet18_25_44      0.5
ISIC2019        resnet18         ISIC2019_resnet18_50_42      0.5
ISIC2019        resnet18         ISIC2019_resnet18_50_43      0.5
ISIC2019        resnet18         ISIC2019_resnet18_50_44      0.5
ISIC2019        resnet18        ISIC2019_resnet18_100_42      0.5
ISIC2019        resnet18        ISIC2019_resnet18_100_43      0.5
ISIC2019        resnet18        ISIC2019_resnet18_100_44      0.5
ISIC2019 efficientnet_b0  ISIC2019_efficientnet_b0_25_42      0.2
ISIC2019 efficientnet_b0  ISIC2019_efficientnet_b0_25_43      0.2
ISIC2019 efficientnet_b0  ISIC2019_efficientnet_b0_25_44      0.2
ISIC2019 efficientnet_b0  ISIC2019_efficientnet_b0_50_42      0.2
ISIC2019 efficientnet_b0  ISIC2019_

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

[ISIC2019] Run 1/1 run_id=ISIC2019_efficientnet_b0_100_44 start model=efficientnet_b0 seed=44 frac=1.00 dropout=0.20 lr=0.0001 device=cuda


[ISIC2019] Run 1/1 run_id=ISIC2019_efficientnet_b0_100_44 e1/25 train=1.9859 train_acc=0.5065 val_loss=1.2950 val_acc=0.6341 val_f1=0.4969 val_auc=0.9187 lr=0.0001 pat=0/5


...

[ISIC2019] Run 1/1 run_id=ISIC2019_efficientnet_b0_100_44 e22/25 train=0.9262 train_acc=0.9821 val_loss=0.9136 val_acc=0.8469 val_f1=0.7677 val_auc=0.9352 lr=5e-05 pat=0/5


[ISIC2019] Run 1/1 run_id=ISIC2019_efficientnet_b0_100_44 e23/25 train=0.9285 train_acc=0.9832 val_loss=0.9239 val_acc=0.8455 val_f1=0.7701 val_auc=0.9404 lr=5e-05 pat=1/5


[ISIC2019] Run 1/1 run_id=ISIC2019_efficientnet_b0_100_44 e24/25 train=0.9275 train_acc=0.9863 val_loss=0.8969 val_acc=0.8551 val_f1=0.7898 val_auc=0.9442 lr=5e-05 pat=0/5


[ISIC2019] Run 1/1 run_id=ISIC2019_efficientnet_b0_100_44 e25/25 train=0.9264 train_acc=0.9860 val_loss=0.8931 val_acc=0.8545 val_f1=0.7786 val_auc=0.9444 lr=5e-05 pat=0/5


[ISIC2019] Run 1/1 run_id=ISIC2019_efficientnet_b0_100_44 done test_acc=0.6886 test_f1=0.5426 test_auc=0.8208
[PROGRESS] ISIC2019: 18/18 complete | 0 pending
[PROGRESS] ALL: 18/36 complete | 18 pending
[PROGRESS] ALL: 17/18 complete | 1 pending


In [ ]:
# C26 - Running Messidor2 experiments

run_messidor_calls()


[DATASET_CALLS] dataset=MESSIDOR2 rows=18
  dataset           model                           run_id  dropout
Messidor2        resnet18         Messidor2_resnet18_25_42     0.25
Messidor2        resnet18         Messidor2_resnet18_25_43     0.25
Messidor2        resnet18         Messidor2_resnet18_25_44     0.25
Messidor2        resnet18         Messidor2_resnet18_50_42     0.25
Messidor2        resnet18         Messidor2_resnet18_50_43     0.25
Messidor2        resnet18         Messidor2_resnet18_50_44     0.25
Messidor2        resnet18        Messidor2_resnet18_100_42     0.25
Messidor2        resnet18        Messidor2_resnet18_100_43     0.25
Messidor2        resnet18        Messidor2_resnet18_100_44     0.25
Messidor2 efficientnet_b0  Messidor2_efficientnet_b0_25_42     0.25
Messidor2 efficientnet_b0  Messidor2_efficientnet_b0_25_43     0.25
Messidor2 efficientnet_b0  Messidor2_efficientnet_b0_25_44     0.25
Messidor2 efficientnet_b0  Messidor2_efficientnet_b0_50_42     0.25
Messid

100%|██████████| 44.7M/44.7M [00:00<00:00, 243MB/s]


[Messidor2] Run 1/18 run_id=Messidor2_resnet18_25_42 start model=resnet18 seed=42 frac=0.25 dropout=0.25 lr=0.0001 device=cuda


[Messidor2] Run 1/18 run_id=Messidor2_resnet18_25_42 e1/25 train=1.9515 train_acc=0.1772 val_loss=1.7199 val_acc=0.0943 val_f1=0.0779 val_auc=0.4679 lr=0.0001 pat=0/5


[Messidor2] Run 1/18 run_id=Messidor2_resnet18_25_42 e2/25 train=1.5075 train_acc=0.3038 val_loss=1.4573 val_acc=0.4528 val_f1=0.2895 val_auc=0.5395 lr=0.0001 pat=0/5


...

[Messidor2] Run 18/18 run_id=Messidor2_efficientnet_b0_100_44 e15/25 train=0.7487 train_acc=0.8833 val_loss=1.1261 val_acc=0.5877 val_f1=0.5320 val_auc=0.8147 lr=0.0001 pat=2/5


[Messidor2] Run 18/18 run_id=Messidor2_efficientnet_b0_100_44 e16/25 train=0.7325 train_acc=0.8927 val_loss=1.1367 val_acc=0.5877 val_f1=0.5409 val_auc=0.8079 lr=0.0001 pat=3/5


[Messidor2] Run 18/18 run_id=Messidor2_efficientnet_b0_100_44 e17/25 train=0.6680 train_acc=0.9401 val_loss=1.1769 val_acc=0.5545 val_f1=0.4506 val_auc=0.8028 lr=5e-05 pat=4/5


[Messidor2] Run 18/18 run_id=Messidor2_efficientnet_b0_100_44 e18/25 train=0.6548 train_acc=0.9732 val_loss=1.1844 val_acc=0.5545 val_f1=0.4901 val_auc=0.8093 lr=5e-05 pat=5/5
[Messidor2] Run 18/18 run_id=Messidor2_efficientnet_b0_100_44 early_stop


[Messidor2] Run 18/18 run_id=Messidor2_efficientnet_b0_100_44 done test_acc=0.5236 test_f1=0.4269 test_auc=0.7078
[PROGRESS] Messidor2: 18/18 complete | 0 pending
[PROGRESS] ALL: 36/36 complete | 0 pending
[PROGRESS] ALL: 0/18 complete | 18 pending


Cleaning smoke test artifacts from Google Drive


In [ ]:
# C28 - Optional cleanup helper for smoke test JSON bundles

# import shutil

# def delete_smoke_test_artifacts():
#     """
#     Delete artifacts that came from smoke test runs.
#     Smoke runs are identified from the single JSON result bundles.
#     """
#     deleted_files = []
#     deleted_dirs = []
#
#     if not os.path.exists(RUN_ARTIFACTS_DIR):
#         print("[CLEANUP] Run_JSON directory does not exist.")
#         return
#
#     result_files = [
#         os.path.join(RUN_ARTIFACTS_DIR, f)
#         for f in os.listdir(RUN_ARTIFACTS_DIR)
#         if f.endswith("_results.json")
#     ]
#
#     for result_path in result_files:
#         try:
#             with open(result_path, "r") as f:
#                 payload = json.load(f)
#
#             if bool(payload.get("metadata", {}).get("smoke_test", False)):
#                 run_id = payload.get("run_id", "")
#
#                 if os.path.isfile(result_path):
#                     os.remove(result_path)
#                     deleted_files.append(result_path)
#
#                 final_csv = payload.get("manifest", {}).get("final_csv", "")
#                 if final_csv and os.path.isfile(final_csv):
#                     os.remove(final_csv)
#                     deleted_files.append(final_csv)
#
#                 gradcam_dir = payload.get("manifest", {}).get("gradcam_dir", "")
#                 if gradcam_dir and os.path.isdir(gradcam_dir):
#                     shutil.rmtree(gradcam_dir)
#                     deleted_dirs.append(gradcam_dir)
#
#         except Exception as e:
#             print(f"[CLEANUP] Failed to process: {result_path} | {repr(e)}")
#
#     print("[CLEANUP] Deleted files:", len(deleted_files))
#     print("[CLEANUP] Deleted dirs:", len(deleted_dirs))
#
# delete_smoke_test_artifacts()

Stopping the Colab runtime after all experiments finish


In [ ]:
# C29 - Stopping the Colab runtime

import os
import time

def stop_runtime(delay_seconds=60):
    """
    Stop the Colab runtime after a short delay.
    Use a delay so logs and Drive sync have time to finish.
    """
    print(f"[SHUTDOWN] Waiting {delay_seconds} seconds before stopping the runtime...")
    time.sleep(delay_seconds)

    print("[SHUTDOWN] Stopping the runtime now.")
    os.kill(os.getpid(), 9)

stop_runtime(delay_seconds=1)


[SHUTDOWN] Waiting 1 seconds before stopping the runtime...
